
#Mirrors and Image Formation using SMT solvers

Working out where a curved mirror puts an image usually means rearranging the mirror
equation for whichever quantity you were not given, and then keeping track of a table
of sign conventions to say what the answer means. In this notebook we will look at SMT
solvers and hand the whole relationship to Z3 at once, unrearranged. We will describe a
mirror and an object, ask where the image goes, and read the signs off the answer. We
will also describe one arrangement that has no answer at all.


**Instructions:**
1. To get started, click on File on the top left and click "Save a copy in Drive."
This will give you an editable version of this document that you can use.
2. If you press `CMD`+`Enter` it runs the cell, and if you press `Shift`+`Enter` it runs the cell and goes to the next one.
3. Make sure you run all cells as you go through the notebook; some cells will not work properly unless the previous one
has been run too.
4. If you disconnect or are inactive for some time you should run all of the cells again.

## 0. Preliminaries (you should run this cell but there is no need to read it)

In [ ]:
!pip install z3-solver
!pip install git+https://github.com/crrivero/FormalMethodsTasting.git#subdirectory=core
from z3 import *
from tofmcore import showSolver
from IPython.display import clear_output
clear_output()

## Encoding constraints in Z3

The goal of this notebook is to teach you about formal methods;
particularly, how you can use existing formal verification tools
(in this case, Z3) to analyze and solve your own problems.
Before we get started, let's look at some basic things we can do with Z3.

### Reals

Let's use Z3 to solve problems involving real numbers. Let's start with something simple: find $x$ such that

$$2x + 5 = 15$$

In [ ]:
# Initialize variables

x = Real('x') # declairing that x is a real number named 'x'

# Initialize Z3 solver
s = Solver()

s.add( 2*x + 5 == 15 ) # add the equation

print(s)
print(s.check())
print(s.model())

Now let's try to check whether that's the only solution. We can do this by adding the following constraint to the solver:

$$x \not= 5$$

If the solver returns "**unsat**" then $x=5$ is the only solution.
Try it yourself by completing the code in the cell below.

In [ ]:
s.add( x == 5 ) # REPLACE THIS LINE
s.check()

## The Mirror Equation

A curved mirror takes light from an object and brings it back to form an image. Where
that image sits, how large it is, and which way up it is are all fixed by two numbers:
the mirror's focal length $f$ and the object's distance from the mirror $d_o$.

The relationship between them is the **mirror equation**:

$$ \frac{1}{f} = \frac{1}{d_i} + \frac{1}{d_o} $$

where $d_i$ is the distance from the mirror to the image.

Each of these quantities carries a sign, and the signs are where the physics lives:

| If | then |
|---|---|
| $d_i > 0$ | the image is **real** (it forms in front of the mirror) |
| $d_i < 0$ | the image is **virtual** (it appears behind the mirror) |
| $h_i > 0$ | the image is **erect** |
| $h_i < 0$ | the image is **inverted** |
| $f > 0$ | the mirror is **concave** |
| $f < 0$ | the mirror is **convex** |

We will hand the mirror equation to Z3 and let it work out whichever of the three
quantities we leave unknown.

One adjustment before we start. Written with fractions, the equation asks Z3 to
divide by variables it is also solving for, and division by zero is a case we would
rather not have to reason about. Multiplying both sides through by $f \, d_i \, d_o$
gives an equivalent equation with no fractions in it:

$$ d_i \, d_o = f \, d_i + f \, d_o $$

In [ ]:
s = Solver()

# focal length, image distance, object distance
f, d_i, d_o = Reals('f d_i d_o')

s.add(d_i*d_o == f*d_i + f*d_o)

showSolver(s)

Let's start with a mirror whose focal length we do not know.

An object is placed 6 cm in front of a mirror, and the image appears to be 1 cm
**behind** it. Behind the mirror means the image is virtual, so $d_i$ is negative.
What is the focal length?

In [ ]:
s.add(d_o == 6)
s.add(d_i == -1)

print(s.check())
print(s.model())
print(f"f = {s.model()[f]} cm")

The solver returns $f = -6/5$, that is, $-1.2$ cm. A negative focal length means the
mirror is **convex** — it bulges outward, and it cannot form a real image at all. That
is consistent with what we told it: we said the image was behind the mirror, and the
solver found the only kind of mirror that does that.

### Magnification

The mirror equation places the image but says nothing about its size. For that we need
the **magnification** $m$, which is the ratio of image height to object height, and
which is also fixed by the two distances:

$$ m = \frac{h_i}{h_o} = -\frac{d_i}{d_o} $$

While we are here, we can also add the mirror's **radius of curvature**, which is
twice the focal length:

$$ R = 2f $$

Let's add all three constraints to the solver we already have, and say the object is
3 cm tall.

In [ ]:
m, h_i, h_o = Reals('m h_i h_o')
R = Real('R')

s.add(m == h_i/h_o)
s.add(m == -d_i/d_o)
s.add(R == 2*f)

s.add(h_o == 3)

print(s.check())
print(f"m = {s.model()[m]}")
print(f"h_i = {s.model()[h_i]} cm")
print(f"R = {s.model()[R]} cm")

The magnification comes out to $1/6$, so the image is 0.5 cm tall: much smaller than
the object. It is positive, which by the table above means the image is **erect**. So
this convex mirror produces an image that is virtual, upright and shrunken — which is
exactly why convex mirrors are the ones used for wing mirrors and for the mirrors hung
in shop corners.

### A configuration with no answer

Now let's put a concave mirror with a focal length of 10 cm in front of an object, and
place the object exactly 10 cm away, right at the focal point.

Nothing about that description sounds impossible, so let's ask the solver where the
image goes.

In [ ]:
s = Solver()

f, d_i, d_o = Reals('f d_i d_o')

s.add(d_i*d_o == f*d_i + f*d_o)

s.add(f == 10)
s.add(d_o == 10)

showSolver(s)
print(s.check())

"unsat" means no assignment of $d_i$ satisfies the constraints, so **there is no image
distance at all**. This is not a failure of the solver; it is the physics. An object
sitting at the focal point sends out rays that leave the mirror parallel to one
another, and parallel rays never meet, so they never converge into an image. The
fractions version of the equation says the same thing more visibly: $1/d_i$ would have
to be zero.

It is worth noticing what just happened. We did not know in advance that this
configuration was degenerate. We described it, asked, and the solver told us the
description has no solution.

### Your turn

A concave mirror has a focal length of 10 cm. An object 4 cm tall is placed 30 cm in
front of it.

Where does the image form, how tall is it, and is it real or virtual, erect or
inverted?

The known quantities and the magnification relationships are written below.
**Replace the marked line** with the mirror equation, in the fraction-free form we
have been using.

In [ ]:
s = Solver()

f, d_i, d_o = Reals('f d_i d_o')
m, h_i, h_o = Reals('m h_i h_o')

# what we know about the mirror and the object
s.add(f == 10)
s.add(d_o == 30)
s.add(h_o == 4)

# the mirror equation
s.add(True) # REPLACE THIS LINE

# magnification
s.add(m == -d_i/d_o)
s.add(m == h_i/h_o)

showSolver(s)
print(s.check())

In [ ]:
solution = s.model()

print(f"d_i = {solution[d_i]} cm")
print(f"m   = {solution[m]}")
print(f"h_i = {solution[h_i]} cm")

Reading those three numbers against the table at the top: $d_i$ is positive, so the
image is **real** and forms 15 cm in front of the mirror. $h_i$ is negative, so the
image is **inverted**. The magnification is $-1/2$, so it is half the height of the
object, 2 cm tall.

Notice that we never rearranged the mirror equation to isolate $d_i$. We wrote down
what we knew, wrote down the relationships, and the only work left was reading the
signs.


###Congratulations! You just used an SMT solver to locate the image formed by a mirror!


####If you'd like to continue your Z3 journey, you can start with this guide to learn more:
https://ericpony.github.io/z3py-tutorial/guide-examples.htm